# U6 — Bloque 2: Inteligencia Artificial Explicable (XAI)

**Métodos de Análisis de Datos I — UNS 2026**

En este notebook aplicamos las principales técnicas de XAI sobre un dataset sintético de rendimiento de jugadores de fútbol.  
Objetivo: predecir si un jugador será convocado a la selección nacional.

Técnicas cubiertas:
- Importancia de variables por permutación
- PDP (Partial Dependence Plot) e ICE (Individual Conditional Expectation)
- SHAP (SHapley Additive exPlanations)
- LIME (Local Interpretable Model-agnostic Explanations)
- Explicaciones contrafácticas (introducción conceptual)


## 1. Preparación: dataset y modelo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# Semilla reproducible
np.random.seed(42)


In [ ]:
# Generación del dataset sintético
n = 400

goles          = np.random.poisson(6, n)
asistencias    = np.random.poisson(4, n)
minutos        = np.random.normal(1500, 400, n).clip(200, 2800)
pases_clave    = np.random.poisson(40, n)
duelos_ganados = np.random.uniform(35, 75, n)
velocidad      = np.random.normal(28, 3, n).clip(20, 36)
faltas_recibidas = np.random.uniform(0.5, 4, n)
edad           = np.random.randint(18, 36, n)
valor_mercado  = np.random.exponential(8, n).clip(0.5, 80)
posicion       = np.random.choice([1, 2, 3, 4], n)  # 1=portero, 2=defensa, 3=medio, 4=delantero

# Variable objetivo: convocatoria (con lógica realista)
score = (
    0.20 * goles +
    0.12 * asistencias +
    0.008 * minutos +
    0.10 * pases_clave +
    0.05 * duelos_ganados +
    0.08 * velocidad +
    0.06 * faltas_recibidas -
    0.08 * edad +
    0.04 * valor_mercado +
    np.random.normal(0, 1.5, n)
)
convocado = (score > score.mean()).astype(int)

df = pd.DataFrame({
    'goles': goles,
    'asistencias': asistencias,
    'minutos': minutos.astype(int),
    'pases_clave': pases_clave,
    'duelos_ganados': duelos_ganados.round(1),
    'velocidad': velocidad.round(1),
    'faltas_recibidas': faltas_recibidas.round(2),
    'edad': edad,
    'valor_mercado': valor_mercado.round(2),
    'posicion': posicion,
    'convocado': convocado
})

print(df.shape)
df.head()


In [ ]:
# División train/test y entrenamiento del Random Forest
X = df.drop(columns='convocado')
y = df['convocado']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

print(classification_report(y_test, rf.predict(X_test)))


## 2. Importancia de variables por permutación

La importancia por permutación mide cuánto cae el rendimiento del modelo cuando los valores de una variable se barajan aleatoriamente.  
Una caída grande → la variable era importante.


In [ ]:
result = permutation_importance(
    rf, X_test, y_test,
    n_repeats=30, random_state=42, scoring='accuracy'
)

importances = pd.Series(result.importances_mean, index=X.columns)
std_imp     = pd.Series(result.importances_std,  index=X.columns)
importances_sorted = importances.sort_values(ascending=True)
std_sorted         = std_imp[importances_sorted.index]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#1e3c78' if v > 0 else '#b0b8cc' for v in importances_sorted]
ax.barh(importances_sorted.index, importances_sorted.values,
        xerr=std_sorted.values, color=colors,
        error_kw=dict(ecolor='gray', capsize=3), height=0.6)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Caída promedio en accuracy (permutación)')
ax.set_title('Importancia de variables por permutación')
plt.tight_layout()
plt.show()


## 3. PDP e ICE

**PDP** (Partial Dependence Plot): muestra el efecto promedio de una variable sobre la predicción.  
**ICE** (Individual Conditional Expectation): una curva por observación — revela heterogeneidad que el PDP puede ocultar.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PDP + ICE para 'goles'
PartialDependenceDisplay.from_estimator(
    rf, X_test, features=['goles'],
    kind='both', subsample=80, random_state=42,
    ice_lines_kw={'color': '#aab8d8', 'alpha': 0.4, 'linewidth': 0.8},
    pd_line_kw={'color': '#1e3c78', 'linewidth': 2.5},
    ax=axes[0]
)
axes[0].set_title('PDP + ICE: goles')
axes[0].set_ylabel('Probabilidad de convocatoria')

# PDP + ICE para 'edad'
PartialDependenceDisplay.from_estimator(
    rf, X_test, features=['edad'],
    kind='both', subsample=80, random_state=42,
    ice_lines_kw={'color': '#aab8d8', 'alpha': 0.4, 'linewidth': 0.8},
    pd_line_kw={'color': '#1e3c78', 'linewidth': 2.5},
    ax=axes[1]
)
axes[1].set_title('PDP + ICE: edad')
axes[1].set_ylabel('Probabilidad de convocatoria')

plt.suptitle('Gráficos de dependencia parcial e ICE', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# PDP 2D: interacción goles x edad
fig, ax = plt.subplots(figsize=(7, 5))
PartialDependenceDisplay.from_estimator(
    rf, X_test, features=[('goles', 'edad')],
    kind='average', ax=ax
)
ax.set_title('PDP 2D: goles × edad')
plt.tight_layout()
plt.show()


## 4. SHAP

Los valores SHAP descomponen cada predicción en contribuciones individuales por variable, con base en la teoría de juegos cooperativos.  
Propiedad clave: **eficiencia** — las contribuciones suman exactamente la predicción menos el valor base.


In [ ]:
# Instalar shap si no está disponible
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap

# Explainer eficiente para Random Forest (TreeSHAP)
explainer    = shap.TreeExplainer(rf)
shap_values  = explainer.shap_values(X_test)

# Para clasificación binaria tomamos la clase 1
sv = shap_values[1] if isinstance(shap_values, list) else shap_values
print(f"Matriz SHAP: {sv.shape}  (observaciones × variables)")


In [ ]:
# Importancia global SHAP (valor absoluto promedio)
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_imp = pd.Series(mean_abs_shap, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(shap_imp.index, shap_imp.values, color='#1e3c78', height=0.6)
ax.set_xlabel('|SHAP| promedio')
ax.set_title('Importancia global SHAP')
plt.tight_layout()
plt.show()


In [ ]:
# Beeswarm SHAP
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(sv, X_test, plot_type='dot', show=False)
plt.title('Beeswarm SHAP: distribución de contribuciones')
plt.tight_layout()
plt.show()


In [ ]:
# Waterfall: explicación de una observación individual
idx = 0  # primer jugador del conjunto de test
print(f"Predicción: {'CONVOCADO' if rf.predict(X_test.iloc[[idx]])[0] == 1 else 'NO CONVOCADO'}")
print(f"Probabilidad: {rf.predict_proba(X_test.iloc[[idx]])[0, 1]:.3f}")
print()

contrib = pd.Series(sv[idx], index=X.columns).sort_values()
colors_wf = ['#c0392b' if v > 0 else '#2980b9' for v in contrib]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(contrib.index, contrib.values, color=colors_wf, height=0.6)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Contribución SHAP')
ax.set_title(f'Waterfall SHAP — jugador #{X_test.index[idx]}')
plt.tight_layout()
plt.show()


## 5. LIME

LIME aproxima el comportamiento del modelo **localmente** alrededor de una observación, ajustando un modelo lineal simple en su vecindad.  
Es más intuitivo para audiencias no técnicas, pero menos estable que SHAP.


In [ ]:
try:
    import lime
    import lime.lime_tabular
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lime', '-q'])
    import lime
    import lime.lime_tabular

explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=X.columns.tolist(),
    class_names=['No convocado', 'Convocado'],
    mode='classification',
    random_state=42
)


In [ ]:
# Explicación LIME para el mismo jugador
exp = explainer_lime.explain_instance(
    X_test.values[idx],
    rf.predict_proba,
    num_features=10
)

# Visualización manual (más control sobre colores)
lime_vals = exp.as_list()
lime_df   = pd.DataFrame(lime_vals, columns=['variable', 'contribucion'])
lime_df   = lime_df.sort_values('contribucion')

colors_lime = ['#c0392b' if v > 0 else '#2980b9' for v in lime_df['contribucion']]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(lime_df['variable'], lime_df['contribucion'], color=colors_lime, height=0.6)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Contribución LIME')
ax.set_title(f'Explicación LIME — jugador #{X_test.index[idx]}')
plt.tight_layout()
plt.show()


In [ ]:
# Comparación SHAP vs. LIME para el mismo jugador
shap_contrib = pd.Series(sv[idx], index=X.columns)
lime_contrib = pd.Series(dict(lime_vals))

comp = pd.DataFrame({
    'SHAP': shap_contrib,
    'LIME': lime_contrib
}).dropna().sort_values('SHAP')

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(comp))
width = 0.35
ax.barh(x_pos - width/2, comp['SHAP'], width, label='SHAP', color='#1e3c78', alpha=0.85)
ax.barh(x_pos + width/2, comp['LIME'], width, label='LIME', color='#e67e22', alpha=0.85)
ax.set_yticks(x_pos)
ax.set_yticklabels(comp.index)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Contribución')
ax.set_title('Comparación SHAP vs. LIME — mismo jugador')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Explicaciones contrafácticas (introducción)

Una explicación contrafáctica responde: **¿qué mínimo cambio habría cambiado la decisión del modelo?**  
Son accionables: no solo explican por qué, sino *qué hay que cambiar*.


In [ ]:
# Búsqueda simple de contrafáctico para un jugador NO convocado
# Estrategia: modificar variables una a una y buscar el umbral de cambio

jugador_nc = X_test[y_test == 0].iloc[0]
print("Jugador NO convocado:")
print(jugador_nc.to_string())
print(f"\nProb. convocatoria original: {rf.predict_proba([jugador_nc])[0, 1]:.3f}")
print()

# Buscar contrafácticos univariados simples
resultados = []
for col in X.columns:
    for delta in np.linspace(-10, 10, 200):
        copia = jugador_nc.copy()
        copia[col] += delta
        prob = rf.predict_proba([copia])[0, 1]
        if prob >= 0.5:
            resultados.append({
                'variable': col,
                'delta': round(delta, 2),
                'valor_original': round(jugador_nc[col], 2),
                'valor_nuevo': round(copia[col], 2),
                'prob_nueva': round(prob, 3)
            })
            break

cf_df = pd.DataFrame(resultados).sort_values('delta', key=abs)
print("Contrafácticos univariados más económicos:")
print(cf_df.head(5).to_string(index=False))


## Cierre

En este notebook aplicamos las principales herramientas de XAI:

| Técnica | Alcance | Complejidad | Estabilidad |
|---|---|---|---|
| Importancia por permutación | Global | Baja | Alta |
| PDP / ICE | Global | Media | Alta |
| SHAP | Global + local | Media-alta | Alta |
| LIME | Local | Media | Media |
| Contrafácticos | Local | Variable | Variable |

**Ninguna técnica reemplaza el juicio contextual.** Las explicaciones son una herramienta para dialogar con el modelo, no verdades absolutas.
